In [10]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_parquet("../data/samples/sample_1m.parquet")

Step 1: Define feature and label

In [6]:
label = "click"

num_cols = ["cost", "cpo", "time_since_last_click"]
cat_cols = [f"cat{i}" for i in range(1, 10)] + ["campaign"]

features = num_cols + cat_cols

Step 2: train / test split (by time)

In [7]:
df = df.sort_values("timestamp")

train = df.iloc[:800000]
test = df.iloc[800000:]

Step 3: Simple baseline (Logistic Regression)

In [8]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ]
)

model = Pipeline([
    ("preprocess", preprocess),
    ("clf", LogisticRegression(max_iter=100))
])

model.fit(train[features], train[label])

pred = model.predict_proba(test[features])[:, 1]

auc = roc_auc_score(test[label], pred)
print("AUC:", auc)

/Users/hanlingjuan/ads-attribution-project/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


AUC: 0.6197601127770406


Step 4: XGBoost model (improved)

In [9]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="auc",
    tree_method="hist"
)

model.fit(train[features], train[label])

pred = model.predict_proba(test[features])[:, 1]

auc = roc_auc_score(test[label], pred)
print("XGB AUC:", auc)

XGB AUC: 0.7477614204564225


Step 5: Feature Engineering

In [11]:
# log transform
df["log_cost"] = np.log1p(df["cost"])
df["log_cpo"] = np.log1p(df["cpo"])

# time feature
df["time_since_last_click"] = df["time_since_last_click"].replace(-1, np.nan)
df["has_prev_click"] = df["time_since_last_click"].notnull().astype(int)

In [12]:
# Re-define features
num_cols = ["log_cost", "log_cpo", "time_since_last_click", "has_prev_click"]
cat_cols = [f"cat{i}" for i in range(1, 10)] + ["campaign"]

features = num_cols + cat_cols

In [13]:
# Re-split
df = df.sort_values("timestamp")

train = df.iloc[:800000]
test = df.iloc[800000:]

In [14]:
# campaign CTR encoding
ctr_map = train.groupby("campaign")["click"].mean()

train["campaign_ctr"] = train["campaign"].map(ctr_map)
test["campaign_ctr"] = test["campaign"].map(ctr_map)

features.append("campaign_ctr")

In [15]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

# Re-run XGBoost
model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="auc",
    tree_method="hist"
)

model.fit(train[features], train["click"])

pred = model.predict_proba(test[features])[:, 1]

auc = roc_auc_score(test["click"], pred)
print("XGB (FE) AUC:", auc)

XGB (FE) AUC: 0.7497469728267072


Step 6: Parameter Tuning (improve AUC)

In [16]:
# Re-run XGBoost
model = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="auc",
    tree_method="hist"
)

model.fit(train[features], train["click"])

pred = model.predict_proba(test[features])[:, 1]

auc = roc_auc_score(test["click"], pred)
print("XGB (FE_2) AUC:", auc)

XGB (FE_2) AUC: 0.7560829066867893


Step 7: Ranking

In [17]:
# score = CTR × value
test["score"] = pred * test["cpo"]

In [18]:
test.sort_values("score", ascending=False).head(20)

,timestamp,uid,campaign,conversion,conversion_timestamp,conversion_id,attribution,click,click_pos,click_nb,...,cat5,cat6,cat7,cat8,cat9,log_cost,log_cpo,has_prev_click,campaign_ctr,score
861231,139156,19632668,7686710,1,546695,3112111,1,1,6,11,...,5824239,1973606,16098334,23998111,358246,0.009815,0.504130,1,0.325287,0.613452
861550,139191,19632668,7686710,1,546695,3112111,1,1,7,11,...,5824239,1973606,16098334,23998111,358246,0.009815,0.504130,1,0.325287,0.613452
860949,139123,19632668,7686710,1,546695,3112111,1,1,5,11,...,5824239,1973606,16098334,23998111,15351053,0.006189,0.504130,1,0.325287,0.585371
815696,134005,210245,7686710,0,-1,-1,0,0,-1,-1,...,5824239,1973606,10003874,20754144,22419790,0.003258,0.504116,1,0.325287,0.564600
802903,132561,4729597,7686710,0,-1,-1,0,1,-1,-1,...,5824239,32440053,32364535,29196072,29520626,0.001343,0.504116,1,0.325287,0.517728
967769,151169,21423334,15661602,0,-1,-1,0,1,-1,-1,...,26611394,28928366,4714295,29196072,6083952,0.004121,0.492763,1,0.328170,0.513583
810229,133381,32295384,7686710,1,293777,15044584,1,1,3,4,...,5824239,1973606,16098334,29841067,32145483,0.001418,0.504116,1,0.325287,0.507440
862814,139329,21175997,7686710,0,-1,-1,0,0,-1,-1,...,5824239,28928366,32364535,29196072,16022563,0.000557,0.504130,1,0.325287,0.500203
848038,137659,4985067,7462178,0,-1,-1,0,1,-1,-1,...,26611394,1973606,23700068,32440044,32145483,0.004096,0.475550,1,0.304843,0.487653
846070,137426,27909093,7462178,0,-1,-1,0,0,-1,-1,...,26611394,1973606,18342753,3225256,6083952,0.002498,0.475498,1,0.304843,0.487597


Step 8: Attribution Modeling

In [22]:
# Last-touch attribution
df["last_touch"] = np.where(
    (df["conversion"] == 1) & (df["click"] == 1) & (df["click_pos"] == df["click_nb"] - 1),
    1.0,
    0.0
)

In [23]:
# Linear multi-touch attribution
df["multi_touch_linear"] = np.where(
    (df["conversion"] == 1) & (df["click"] == 1) & (df["click_nb"] > 0),
    1 / df["click_nb"],
    0.0
)

In [24]:
attr_compare = df.groupby("campaign")[["last_touch", "multi_touch_linear"]].mean()
attr_compare.head(20)

,last_touch,multi_touch_linear
campaign,,
73322,0.038462,0.038462
73325,0.033426,0.034408
73327,0.110465,0.102762
73328,0.058473,0.060253
83677,0.007213,0.006251
289466,0.027586,0.017471
336258,0.009524,0.003810
408759,0.095038,0.095711
442617,0.005739,0.004048


In [25]:
attr_compare["diff"] = attr_compare["multi_touch_linear"] - attr_compare["last_touch"]
attr_compare.sort_values("diff", ascending=False).head(20)

,last_touch,multi_touch_linear,diff
campaign,,,
29036280,0.000000,0.025000,0.025000
8904991,0.020619,0.040300,0.019681
24045153,0.011111,0.030788,0.019677
15518374,0.048387,0.061982,0.013594
29531976,0.111111,0.123737,0.012626
10810197,0.000000,0.012372,0.012372
29531977,0.086957,0.097826,0.010870
4084327,0.028986,0.039855,0.010870
19000595,0.017143,0.027823,0.010680


In [26]:
attr_compare.sort_values("diff", ascending=True).head(20)

,last_touch,multi_touch_linear,diff
campaign,,,
31427832,0.142857,0.071429,-0.071429
13442441,0.076923,0.038462,-0.038462
9551057,0.058824,0.032680,-0.026144
21005924,0.039216,0.016340,-0.022876
21898401,0.052632,0.031579,-0.021053
8403853,0.029412,0.016176,-0.013235
30534043,0.098413,0.085685,-0.012728
11765780,0.037879,0.025152,-0.012727
30487330,0.089362,0.076738,-0.012624


In [28]:
campaign_budget_view = df.groupby("campaign")[["last_touch", "multi_touch_linear", "cpo", "cost"]].mean()
campaign_budget_view

,last_touch,multi_touch_linear,cpo,cost
campaign,,,,
73322,0.038462,0.038462,0.018604,0.000109
73325,0.033426,0.034408,0.008643,0.000065
73327,0.110465,0.102762,0.010318,0.000217
73328,0.058473,0.060253,0.007611,0.000077
83677,0.007213,0.006251,0.320235,0.000327
...,...,...,...,...
32398755,0.009832,0.008635,0.208868,0.000270
32398758,0.014815,0.010246,0.283775,0.000229
32405311,0.010989,0.009343,0.191614,0.000572


Step 9: Budget Allocation / ROI

In [29]:
campaign_budget_view["roi_proxy"] = (
    campaign_budget_view["multi_touch_linear"] / campaign_budget_view["cpo"]
)

campaign_budget_view.sort_values("roi_proxy", ascending=False).head(20)

,last_touch,multi_touch_linear,cpo,cost,roi_proxy
campaign,,,,,
5544859,0.231588,0.229358,0.004000,0.000130,57.339553
2869134,0.181865,0.179712,0.004000,0.000059,44.927989
24843272,0.169054,0.166347,0.004000,0.000091,41.586768
6810192,0.161017,0.159887,0.004338,0.000188,36.859791
15506599,0.151020,0.146429,0.004000,0.000096,36.607143
9100692,0.151257,0.151172,0.004311,0.000144,35.066155
3073305,0.138756,0.139952,0.004000,0.000044,34.988038
16184517,0.130942,0.129423,0.004095,0.000185,31.602341
29531983,0.131351,0.125134,0.004000,0.000039,31.283561


In [30]:
top = campaign_budget_view.sort_values("roi_proxy", ascending=False).head(10)
bottom = campaign_budget_view.sort_values("roi_proxy", ascending=True).head(10)

top, bottom

(          last_touch  multi_touch_linear       cpo      cost  roi_proxy
 campaign                                                               
 5544859     0.231588            0.229358  0.004000  0.000130  57.339553
 2869134     0.181865            0.179712  0.004000  0.000059  44.927989
 24843272    0.169054            0.166347  0.004000  0.000091  41.586768
 6810192     0.161017            0.159887  0.004338  0.000188  36.859791
 15506599    0.151020            0.146429  0.004000  0.000096  36.607143
 9100692     0.151257            0.151172  0.004311  0.000144  35.066155
 3073305     0.138756            0.139952  0.004000  0.000044  34.988038
 16184517    0.130942            0.129423  0.004095  0.000185  31.602341
 29531983    0.131351            0.125134  0.004000  0.000039  31.283561
 29531976    0.111111            0.123737  0.004011  0.000081  30.850445,
           last_touch  multi_touch_linear       cpo      cost  roi_proxy
 campaign                                         

In [32]:
campaign_budget_view["ctr"] = df.groupby("campaign")["click"].mean()
campaign_budget_view

,last_touch,multi_touch_linear,cpo,cost,roi_proxy,ctr
campaign,,,,,,
73322,0.038462,0.038462,0.018604,0.000109,2.067420,0.326923
73325,0.033426,0.034408,0.008643,0.000065,3.981026,0.256267
73327,0.110465,0.102762,0.010318,0.000217,9.959333,0.252907
73328,0.058473,0.060253,0.007611,0.000077,7.916646,0.313842
83677,0.007213,0.006251,0.320235,0.000327,0.019520,0.295291
...,...,...,...,...,...,...
32398755,0.009832,0.008635,0.208868,0.000270,0.041344,0.375256
32398758,0.014815,0.010246,0.283775,0.000229,0.036107,0.316667
32405311,0.010989,0.009343,0.191614,0.000572,0.048761,0.406593


Step 10: Uplift Modeling

In [36]:
df["treatment"] = (df["click"] == 1).astype(int)

df = df.sort_values("timestamp")

train = df.iloc[:800000]
test = df.iloc[800000:]

uplift = P(conversion | treated) - P(conversion | control)

In [39]:
# Rebuild campaign_ctr on the NEW train/test
ctr_map = train.groupby("campaign")["click"].mean()
train["campaign_ctr"] = train["campaign"].map(ctr_map)
test["campaign_ctr"] = test["campaign"].map(ctr_map)

# Rebuild features
num_cols = ["log_cost", "log_cpo", "time_since_last_click", "has_prev_click", "campaign_ctr"]
cat_cols = [f"cat{i}" for i in range(1, 10)] + ["campaign"]
features = num_cols + cat_cols

In [40]:
# Two training groups
train_t = train[train["treatment"] == 1]
train_c = train[train["treatment"] == 0]

In [41]:
from xgboost import XGBClassifier

# Train two models
model_t = XGBClassifier(n_estimators=100, max_depth=6)
model_c = XGBClassifier(n_estimators=100, max_depth=6)

model_t.fit(train_t[features], train_t["conversion"])
model_c.fit(train_c[features], train_c["conversion"])

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [42]:
# Predict uplift
p_t = model_t.predict_proba(test[features])[:, 1]
p_c = model_c.predict_proba(test[features])[:, 1]

uplift = p_t - p_c

In [43]:
# Use uplift for ranking
test["uplift_score"] = uplift

test.sort_values("uplift_score", ascending=False).head(20)

,timestamp,uid,campaign,conversion,conversion_timestamp,conversion_id,attribution,click,click_pos,click_nb,...,cat9,log_cost,log_cpo,has_prev_click,last_touch,multi_touch,multi_touch_linear,treatment,campaign_ctr,uplift_score
848549,137723,31873644,24843272,0,-1,-1,0,0,-1,-1,...,8661620,0.000039,0.003992,1,0.0,0,0.000000,0,0.259928,0.996800
848558,137724,31873644,24843272,0,-1,-1,0,0,-1,-1,...,8661620,0.000039,0.003992,1,0.0,0,0.000000,0,0.259928,0.996800
800838,132327,272199,24843272,1,135119,5698437,1,1,0,1,...,15351053,0.000623,0.003992,1,1.0,1,1.000000,1,0.259928,0.995935
929783,147148,5847226,24843272,1,237004,6637428,1,1,0,1,...,8661620,0.000034,0.003992,1,1.0,1,1.000000,1,0.259928,0.995356
807209,133041,421347,24843272,1,141158,15973913,1,1,0,1,...,9491354,0.000505,0.003992,1,1.0,1,1.000000,1,0.259928,0.995027
822177,134740,18513265,16184517,1,134766,15776035,1,1,0,1,...,29520629,0.021661,0.004168,1,1.0,1,1.000000,1,0.240866,0.994845
892501,142839,15955928,5544859,0,-1,-1,0,0,-1,-1,...,15351053,0.000546,0.003992,1,0.0,0,0.000000,0,0.541348,0.994355
997359,154264,27562913,24843272,0,-1,-1,0,0,-1,-1,...,18291872,0.000038,0.003992,1,0.0,0,0.000000,0,0.259928,0.994134
822466,134771,18513265,16184517,0,-1,-1,0,0,-1,-1,...,358249,0.025272,0.004168,1,0.0,0,0.000000,0,0.240866,0.994129
822954,134826,18513265,16184517,1,134833,4025870,1,1,0,1,...,22419795,0.025266,0.004168,1,1.0,1,1.000000,1,0.240866,0.993108


In [44]:
# Compare CTR vs Uplift
test["ctr_score"] = pred

test[["ctr_score", "uplift_score"]].corr()

,ctr_score,uplift_score
ctr_score,1.000000,0.022506
uplift_score,0.022506,1.000000


In [45]:
# Conversion via uplift
test_sorted = test.sort_values("uplift_score", ascending=False)

top_10 = test_sorted.head(int(len(test)*0.1))
bottom_10 = test_sorted.tail(int(len(test)*0.1))

top_10["conversion"].mean(), bottom_10["conversion"].mean()

(np.float64(0.22335), np.float64(0.00215))

In [47]:
# Ranking via uplift
test_sorted = test.sort_values("uplift_score", ascending=False)

# cumulative conversion gain
test_sorted["cum_conv"] = test_sorted["conversion"].cumsum()

test_sorted

,timestamp,uid,campaign,conversion,conversion_timestamp,conversion_id,attribution,click,click_pos,click_nb,...,log_cpo,has_prev_click,last_touch,multi_touch,multi_touch_linear,treatment,campaign_ctr,uplift_score,ctr_score,cum_conv
848549,137723,31873644,24843272,0,-1,-1,0,0,-1,-1,...,0.003992,1,0.0,0,0.0,0,0.259928,0.996800,0.423915,0
848558,137724,31873644,24843272,0,-1,-1,0,0,-1,-1,...,0.003992,1,0.0,0,0.0,0,0.259928,0.996800,0.094565,0
800838,132327,272199,24843272,1,135119,5698437,1,1,0,1,...,0.003992,1,1.0,1,1.0,1,0.259928,0.995935,0.289214,1
929783,147148,5847226,24843272,1,237004,6637428,1,1,0,1,...,0.003992,1,1.0,1,1.0,1,0.259928,0.995356,0.185419,2
807209,133041,421347,24843272,1,141158,15973913,1,1,0,1,...,0.003992,1,1.0,1,1.0,1,0.259928,0.995027,0.161363,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
914330,145437,1033114,12947794,0,-1,-1,0,1,-1,-1,...,0.381378,1,0.0,0,0.0,1,0.239063,0.000467,0.233552,9294
927636,146915,13659722,28137208,0,-1,-1,0,0,-1,-1,...,0.342912,1,0.0,0,0.0,0,0.361815,0.000450,0.232818,9294
823212,134852,31965457,28137208,0,-1,-1,0,0,-1,-1,...,0.311961,0,0.0,0,0.0,0,0.361815,0.000430,0.320910,9294
993923,153912,22294515,28137208,0,-1,-1,0,0,-1,-1,...,0.342912,0,0.0,0,0.0,0,0.361815,0.000416,0.193900,9294
